# 04 - Entrenamiento y Optimización

**Objetivo:** entrenar varios modelos de clasificación, evaluarlos con el **F1-Score** sobre el conjunto de validación, y elegir el mejor.

Modelos a probar: Árbol de Decisión, KNN, Random Forest y Gradient Boosting.


In [1]:
import pandas as pd

# Cargo los datos ya procesados ya que funcionan de forma independiente
X_train = pd.read_csv("../data/processed/X_train.csv")
X_val   = pd.read_csv("../data/processed/X_val.csv")
y_train = pd.read_csv("../data/processed/y_train.csv")["smoking"]
y_val   = pd.read_csv("../data/processed/y_val.csv")["smoking"]

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:  ", y_val.shape)


X_train: (40000, 24)
X_val:   (10000, 24)
y_train: (40000,)
y_val:   (10000,)


In [2]:
from sklearn.tree import DecisionTreeClassifier

# 1) Creo el modelo  (todavía no aprendió nada)
arbol = DecisionTreeClassifier(random_state=42)


arbol.fit(X_train, y_train)

print("entrenado")


entrenado


In [3]:
from sklearn.metrics import f1_score, accuracy_score, classification_report


y_pred = arbol.predict(X_val)

# Comparo las predicciones con las respuestas reales (y_val)
print("F1-Score (clase 1):", round(f1_score(y_val, y_pred), 4))
print("Accuracy:          ", round(accuracy_score(y_val, y_pred), 4))
print()
print(classification_report(y_val, y_pred))


F1-Score (clase 1): 0.653
Accuracy:           0.7461

              precision    recall  f1-score   support

           0       0.80      0.80      0.80      6334
           1       0.65      0.65      0.65      3666

    accuracy                           0.75     10000
   macro avg       0.73      0.73      0.73     10000
weighted avg       0.75      0.75      0.75     10000



Accuracy = 0,746 , acerto el 74,6% de las veces en total
F1 (clase 1) = 0,653, más bajo.
El F1 se concentra en los fumadores, que es lo que cuesta. 

In [4]:
# Comparo el F1 en entrenamiento vs validación
y_pred_train = arbol.predict(X_train)
print("F1 en ENTRENAMIENTO:", round(f1_score(y_train, y_pred_train), 4))
print("F1 en VALIDACIÓN:   ", round(f1_score(y_val, y_pred), 4))


F1 en ENTRENAMIENTO: 1.0
F1 en VALIDACIÓN:    0.653


In [5]:
from sklearn.ensemble import RandomForestClassifier

# Creo y entreno un bosque de 200 arboles
bosque = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
bosque.fit(X_train, y_train)


y_pred_rf = bosque.predict(X_val)
print("F1 Random Forest (validación):", round(f1_score(y_val, y_pred_rf), 4))


F1 Random Forest (validación): 0.737


In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Boosting: árboles en secuencia que corrigen los errores del anterior

boosting = HistGradientBoostingClassifier(random_state=42)
boosting.fit(X_train, y_train)

y_pred_gb = boosting.predict(X_val)
print("F1 Gradient Boosting (validación):", round(f1_score(y_val, y_pred_gb), 4))


F1 Gradient Boosting (validación): 0.6953


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Pipeline 
knn = Pipeline([
    ("escalador", StandardScaler()),
    ("modelo", KNeighborsClassifier(n_neighbors=15))
])

knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_val)
print("F1 KNN (validación):", round(f1_score(y_val, y_pred_knn), 4))


F1 KNN (validación): 0.6336


In [8]:
from sklearn.model_selection import GridSearchCV

# Las "perillas" que quiero probar (y sus valores)
grilla = {
    "learning_rate": [0.05, 0.1],
    "max_iter": [200, 400],
    "max_leaf_nodes": [31, 63]
}

busqueda = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    grilla,
    scoring="f1",   # ¡optimiza nuestra métrica: el F1 de la clase 1!
    cv=3,           # validación cruzada de 3 pliegues
    n_jobs=-1
)
busqueda.fit(X_train, y_train)

print("Mejores perillas:", busqueda.best_params_)
print("Mejor F1 (cross-val):", round(busqueda.best_score_, 4))


Mejores perillas: {'learning_rate': 0.05, 'max_iter': 200, 'max_leaf_nodes': 63}
Mejor F1 (cross-val): 0.6955


In [9]:
# boosting optimizado
mejor_boosting = busqueda.best_estimator_
y_pred_bopt = mejor_boosting.predict(X_val)
print("F1 Boosting optimizado (validación):", round(f1_score(y_val, y_pred_bopt), 4))


F1 Boosting optimizado (validación): 0.6973


In [10]:
import numpy as np

# 1) La probabilidad de fumar que el Random Forest le asigna a cada persona
probs = bosque.predict_proba(X_val)[:, 1]


mejor_umbral = 0.5
mejor_f1 = 0
for u in np.arange(0.1, 0.9, 0.01):
    pred_u = (probs >= u).astype(int)     # predigo 1 si la probabilidad supera el umbral u
    f1_u = f1_score(y_val, pred_u)
    if f1_u > mejor_f1:
        mejor_f1 = f1_u
        mejor_umbral = u

print("Mejor umbral:", round(mejor_umbral, 2))
print("F1 con ese umbral:", round(mejor_f1, 4))
print("(Con el umbral 0.5 de fábrica, el F1 era 0.737)")


Mejor umbral: 0.39
F1 con ese umbral: 0.7546
(Con el umbral 0.5 de fábrica, el F1 era 0.737)


In [11]:
import joblib

# Guardo el modelo ganador y el umbral elegido
joblib.dump(bosque, "../models/random_forest.joblib")
joblib.dump(0.39, "../models/umbral.joblib")
print("✅ Guardado en models/: random_forest.joblib y umbral.joblib")


✅ Guardado en models/: random_forest.joblib y umbral.joblib


## Conclusiones del entrenamiento

Entrenamos y comparamos 4 modelos (F1 de la clase 1 en validación):

| Modelo | F1 |
|---|---|
| Árbol de Decisión | 0,653 |
| KNN | 0,634 |
| Gradient Boosting (tuneado) | 0,697 |
| **Random Forest** | **0,737** |

- El **Random Forest** fue el mejor. Un solo árbol hacía overfitting (F1 entrenamiento = 1,0 vs validación = 0,653); el bosque generaliza mejor.
- Optimizamos el **umbral de decisión** a **0,39**, subiendo el F1 de 0,737 a **0,7546**.
- **Modelo final elegido: Random Forest + umbral 0,39.**


In [12]:
import joblib

# Guardo el modelo COMPRIMIDO para que ocupe menos y entre en GitHub
joblib.dump(bosque, "../models/random_forest.joblib", compress=3)
joblib.dump(0.39, "../models/umbral.joblib")
print("✅ Guardado comprimido en models/")


✅ Guardado comprimido en models/
